In [1]:
model_name = 'qwen' #'qwen', 'intern' or 'gemma'

#### Import and utilities

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from ast import literal_eval # to get dict/list from csv
import hydra
import os
import matplotlib.pyplot as plt
from scipy import stats
from transformers.utils import ModelOutput
from transformers import GenerationConfig

import pyrootutils,sys
pyrootutils.setup_root('.',cwd=True,pythonpath=False)
sys.path.append('./src')

In [ ]:
hydra.core.global_hydra.GlobalHydra.instance().clear()
hydra.initialize("config")
cfg = hydra.compose("activations_gen", overrides=[dict(qwen='+experiment=bal_qwen7b',
                                                       intern='+experiment=bal_internvl8b',
                                                       gemma='+experiment=bal_gemma12b')[model_name],
                                                  'probe=attn_single'
                                                  ])
probelabel = 'mmp' #layer used to read hidden activations (defined in config)

/tmp/ipykernel_3460903/165392830.py:2: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize("config")


### Load model

In [ ]:
model = hydra.utils.instantiate(cfg.model)

## Generation config
if 'Intern' in cfg.model.model_name:
    generation_kwargs = dict(generation_config=GenerationConfig( 
    max_new_tokens= 4, 
    do_sample= False, #avoid sampling outputs (greedy choice)
    return_dict_in_generate= True,  #outputs is a dictionary, following args specify its contents
    output_attentions= False, # Passed to all submodels, needed to use hooks
    output_hidden_states= False,
    output_logits= True,  # These are handy
    output_scores= False,
    eos_token_id=92542
    ) )
    # I'll move this to the model class sooner or later
    def getstring(model,output,input_len):
      return model.processor.decode(output['sequences'][0])
else:
    generation_kwargs = { 
      "max_new_tokens": 4,
      "do_sample": False, #avoid sampling outputs (greedy choice)
      "return_dict_in_generate": True,  #outputs is a dictionary, following args specify its contents
      "output_attentions": False, # Passed to all submodels, needed to use hooks
      "output_hidden_states": False,
      "output_logits": True,  # These are handy
      "output_scores": False,
    }
    def getstring(model,output,input_len):
        return model.processor.decode(output['sequences'][0][input_len:])

### Query the model

In [ ]:
for i, row in intvl_results_sim.iterrows():
    #if i==20:#GUARD
    #    break#remove this guard to run over all images
    if i%100==0:
        print(i,end=' ')
    img_path_1 = COLOR_REFERENCE_DATASET+str(row['ID']).zfill(4)+'.png'
    img_path_2 = COLOR_SIMILARITY_DATASET+str(row['ID']).zfill(4)+'.png'
    query = make_query(int(row['N_STENCILS']))
    response = ask_intvl_2imgs(intvl_model,intvl_processor,intvl_vocab,img_path_1,img_path_2,query)
    
    answer = response['A'].upper().strip()
    if len(answer.split()) > 1 or answer not in all_letters:
        print('UNEXPECTED ANSWER '+str(row['ID'])+'--------->'+answer)
    intvl_results_sim.loc[i,'model_answer']=answer
    for s in all_letters:
        intvl_results_sim.loc[i,s]=response['logits'][find_value_intvl(intvl_vocab,s)].item()